# Agente Car con ambiente Grid

Código con ejemplo básico de agente para simular un automóvil. Se deben implementar funciones para interacción, evitar colisiones y permitir vueltas y reglas requeridas por el reto.

In [1]:
!pip install agentpy
# Model design
import agentpy as ap
import numpy as np, random

# Visualization
import matplotlib.pyplot as plt
import IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 778.9/778.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 2.7 MB/s eta 0:00:00


## Definición del modelo

In [3]:
'''
'''
class Car(ap.Agent):
    """ An agent with velocity in a discrete space """
    def setup(self):
        # Agregamos referencia a ambiente
        self.city = self.model.city

    def setup_velocity(self):
        # De acuerdo al carril inicial, se asigna velocidad (dirección)
        self.velocity = np.array([1,0]) if self.get_position()[0] == 1 else np.array([0,1])

    # Obtenemos la posición del agente
    def get_position(self):
        return self.city.positions[self]

    # Se actualiza la posición de acuerdo a la velocidad
    def update_position(self):
        self.city.move_by(self, self.velocity)

        # Aleatoriamente acelera o frena.
        #if np.random.rand() < .5:
        #    self.acelera() if np.random.rand() < .5 else self.frena()
        #self.city.move_by(self, np.round(np.array(self.velocity)).astype(int))

    # Acelera un 10% de la velocidad actual
    def acelera(self):
        self.velocity[0] = 1.10 * self.velocity[0]
        self.velocity[1] = 1.10 * self.velocity[1]

    # Frena un 10% de la velocidad actual
    def frena(self):
        self.velocity[0] = .9 * self.velocity[0]
        self.velocity[1] = .9 * self.velocity[1]

'''
Class City. It keeps information of selected cells:
  - turnright_cell: an agent can turn right in this cell
  - turnleft_cell: an agent can turn left in this cell
  - bus_stops: a dictionary linking a specific coordinate to a number
'''
class City(ap.Grid):

    def setup(self):
        # Celdas centrales permiten vuelta a la derecha e izquierda
        size = self.p.size
        self.turnright_cell = np.array((size // 2, size // 2-1))
        self.turnleft_cell = np.array((size // 2-1, size // 2))

        # Coordenadas de parada de autobús.
        self.bus_stops = dict(zip(self.p.stops, [0] * len(self.p.stops)))

    def update(self):
        # Randomly add passenger number for each stop
        for stop in self.bus_stops.keys():
            if np.random.rand() < .1:
                self.bus_stops[stop] = np.random.randint(0, 10)


'''
Class CarModel. Implements interaction between agent car and city environment.
'''
class CarModel(ap.Model):
    def setup(self):
        # Inicializamos parametros del modelo.
        size = self.p.size
        self.city = City(self, shape=(size, size))


    def step(self):
        # De acuerdo a la probabilidad de aparición, se genera un carro.
        if np.random.rand() < self.p.prob:
            # Generación de un valor aleatorio para seleccionar el carril
            lane = np.random.choice(range(self.p.lanes))
            half = self.p.size // 2
            # Posiciona al agente al inicio del carril
            pos = [np.array([1, half - lane])] if np.random.rand() < .5 else [np.array([half-lane, 1])]
            cars = ap.AgentList(self, 1, Car)
            self.city.add_agents(cars, positions=pos)
            # Velocidad inicial (con dirección), de acuerdo al carril seleccionado
            cars.setup_velocity()


        # Eliminar agentes que salen de escena
        remove = []

        # Se cambiará la ruta de forma aleatoria a aquellos agentes en la celda adecuada.
        # Esto podría hacerse por el mismo agente, considerando un objetivo
        for car in self.city.agents:
            pos = car.get_position()
            # Agente en la orilla del grid
            if pos[0] <= 0 or pos[0] >= self.p.size-1 or pos[1] <= 0 or pos[1] >= self.p.size-1:
                remove.append(car)
            # Agente en celdas para vuelta, elige aleatoriamente
            if np.array_equal(pos, self.city.turnright_cell) and car.velocity[1] == 1:
                car.velocity = np.array(random.choice([[0, 1], [1, 0]]))
            if np.array_equal(pos, self.city.turnleft_cell) and car.velocity[0] == 1:
                car.velocity = np.array(random.choice([[0, 1], [1, 0]]))

        self.city.remove_agents(remove)

        # Actualizar posición
        self.city.update()
        self.city.agents.update_position()



## Visualización

In [4]:
def animation_plot_single(m, ax):
    ax.set_title(f"Step t={m.t}")

    # Pintamos las celdas de vuelta
    ax.scatter(*m.city.turnright_cell, s=30, marker='>', c='green')
    ax.scatter(*m.city.turnleft_cell, s=30, marker='^', c='green')

    arriba = np.array(list(map(lambda a: a.get_position(), filter(lambda x: x.velocity[0] == 0, m.city.agents)))).T
    if len(arriba) > 0:
        ax.scatter(*arriba, s=30, marker='^', c='black')
    derecha = np.array(list(map(lambda a: a.get_position(), filter(lambda x: x.velocity[1] == 0, m.city.agents)))).T
    if len(derecha) > 0:
        ax.scatter(*derecha, s=30, marker='>', c='black')


    # Pintamos paradas de bus y numero de personas
    stops = list(m.city.bus_stops.keys())
    for stop in stops:
        x, y = stop
        ax.text(x, y, str(m.city.bus_stops[stop]), fontsize=12)

    stops = np.array(stops)
    ax.scatter(*stops, s=30, marker='8', c='red')


    ax.set_xlim(0, m.p.size-1)
    ax.set_ylim(0, m.p.size-1)
    #ax.set_axis_off()

def animation_plot(m, p):
    fig = plt.figure(figsize=(7,7))
    ax = fig.add_subplot(111)
    animation = ap.animate(m(p), fig, ax, animation_plot_single)
    return IPython.display.HTML(animation.to_jshtml(fps=5))

## Simulación

To run a simulation, we define a dictionary with our parameters:

In [5]:
'''
Parameters:
    - lanes: number of lanes in each street
'''

parameters = {
    'size': 30,
    'lanes': 2,
    'steps': 100,
    'seed': 123,
    'prob': .1,
    'stops': [(20, 15), (14, 20)]}

animation_plot(CarModel, parameters)

In [6]:
'''
Parameters:
    - lanes: number of lanes in each street
'''

parameters = {
    'size': 30,
    'lanes': 2,
    'steps': 100,
    'seed': 123,
    'prob': 1,
    'stops': [(20, 15), (14, 20)]}

animation_plot(CarModel, parameters)